# Maestra Grupo B - Problema 04 FE historico

Correr en orden: celda 1 (configurar), 2 (verificar), 3 (ejecutar), 4 (consolidar).

In [16]:
library(data.table)
library(yaml)

# ==========================================
# NOTEBOOK MAESTRA - GRUPO B
# ==========================================

# ---- LO UNICO QUE SE TOCA ----
CONFIGS    <- c("E4")   # luego: c("E0","E1","E2","E3","E4")
N_SEMILLAS <- 10         # 1 = primera corrida real (valida todo el circuito); luego 10

SEMILLAS <- c(
  770503, 216751, 126233, 877577, 388403,
  308563, 344785, 618973, 245039, 552707
)
semillas_a_correr <- SEMILLAS[seq_len(N_SEMILLAS)]

# Resultados: bucket persistente
BASE <- "/content/buckets/b1/exp/GrupoB"
stopifnot(dir.exists("/content/buckets/b1"))
dir.create(file.path(BASE, "params"), recursive = TRUE, showWarnings = FALSE)

cat("Configuraciones:", CONFIGS, "\n")
cat("Semillas a correr:", semillas_a_correr, "\n")

Configuraciones: E4 
Semillas a correr: 770503 216751 126233 877577 388403 308563 344785 618973 245039 552707 


In [17]:
# ==========================================
# LOCALIZAR EL WORKFLOW Y VERIFICAR EL ENTORNO
# ==========================================

# ruta de la carpeta P04_FEhistorico_GrupoB
CARPETA_NOTEBOOKS <- "/home/ds/labo2026v/P04_FEhistorico_GrupoB"

if (!nzchar(CARPETA_NOTEBOOKS)) {
  cand <- Sys.glob("/content/buckets/*/ExpColaborativos/P04_FEhistorico_GrupoB")
  if (length(cand) == 1) CARPETA_NOTEBOOKS <- cand
}
if (!nzchar(CARPETA_NOTEBOOKS))
  stop("No encontre la carpeta del workflow. Completar CARPETA_NOTEBOOKS a mano.")

WF_ORIGINAL  <- file.path(CARPETA_NOTEBOOKS, "02_WorkFlow_GrupoB.ipynb")
WF_CONGELADO <- file.path(CARPETA_NOTEBOOKS, "02_WorkFlow_GrupoB_CONGELADO.ipynb")
stopifnot(file.exists(WF_ORIGINAL))

# Se ejecuta una copia congelada: editar el workflow mientras corre no contamina las semillas.
# Si corregis algo despues de congelar, BORRA el congelado para que se regenere.
if (!file.exists(WF_CONGELADO)) {
  file.copy(WF_ORIGINAL, WF_CONGELADO)
  cat("Copia congelada creada:", WF_CONGELADO, "\n")
}
ARCHIVO_WORKFLOW <- WF_CONGELADO
cat("Workflow a ejecutar:", ARCHIVO_WORKFLOW, "\n")

# Jupyter y kernel de R
stopifnot(nzchar(Sys.which("jupyter")))
ks <- system2(Sys.which("jupyter"), c("kernelspec", "list"), stdout = TRUE)
print(ks)
stopifnot(any(grepl("^\\s*ir\\s", ks)))

Workflow a ejecutar: /home/ds/labo2026v/P04_FEhistorico_GrupoB/02_WorkFlow_GrupoB_CONGELADO.ipynb 
[1] "Available kernels:"                                              
[2] "  python3       /home/ds/.venv/share/jupyter/kernels/python3"    
[3] "  ir            /home/ds/.local/share/jupyter/kernels/ir"        
[4] "  julia-1.10    /home/ds/.local/share/jupyter/kernels/julia-1.10"


In [18]:
# ==========================================
# EJECUCION: semilla afuera, configuracion adentro
# (asi, si falta tiempo, hay N semillas completas para TODAS las configuraciones)
# ==========================================

ARCHIVO_TIEMPOS <- file.path(BASE, "tiempos_corridas.csv")
fallos <- 0

for (semilla in semillas_a_correr) {
  for (cfg in CONFIGS) {

    id <- paste0(cfg, "_S", semilla)
    carpeta <- file.path(BASE, cfg, id)

    # ya completada (existe PARAM.yml): no repetir
    if (file.exists(file.path(carpeta, "PARAM.yml"))) {
      cat("Ya completada:", id, "\n"); next
    }

    # corrida incompleta de un intento anterior: limpiar para no mezclar archivos
    if (dir.exists(carpeta)) unlink(list.files(carpeta, full.names = TRUE), recursive = TRUE)
    dir.create(carpeta, recursive = TRUE, showWarnings = FALSE)

    archivo_param <- file.path(BASE, "params", paste0(id, ".rds"))
    saveRDS(list(
      configuracion      = cfg,
      semilla_primigenia = semilla,
      modo_prueba        = FALSE,
      id_ejecucion       = id
    ), archivo_param)

    Sys.setenv(GRUPOB_PARAMETROS = archivo_param)
    cat("\nINICIANDO:", id, format(Sys.time()), "\n")
    t0 <- Sys.time()

    estado <- system2(
      Sys.which("jupyter"),
      args = c("nbconvert", "--to", "notebook", "--execute",
               "--ExecutePreprocessor.timeout=-1",
               "--output", paste0(id, "_ejecutado.ipynb"),
               "--output-dir", shQuote(carpeta),
               shQuote(ARCHIVO_WORKFLOW)),
      stdout = file.path(carpeta, "salida.log"),
      stderr = file.path(carpeta, "errores.log")
    )
    Sys.unsetenv("GRUPOB_PARAMETROS")

    minutos <- round(as.numeric(difftime(Sys.time(), t0, units = "mins")), 1)
    cat(id, "| estado:", estado, "| minutos:", minutos, "\n")

    # registro de tiempos (para el criterio de costo computacional)
    fwrite(data.table(id = id, cfg = cfg, semilla = semilla, estado = estado,
                      minutos = minutos, fin = format(Sys.time())),
           ARCHIVO_TIEMPOS, append = file.exists(ARCHIVO_TIEMPOS))

    fallos <- if (identical(as.integer(estado), 0L)) 0 else fallos + 1
    if (fallos >= 2) stop("2 fallos consecutivos. Revisar errores.log en ", carpeta)
  }
}
cat("\nFINALIZO\n")

Ya completada: E4_S770503 

INICIANDO: E4_S216751 2026-11-01 09:53:36 
E4_S216751 | estado: 0 | minutos: 132.9 

INICIANDO: E4_S126233 2026-11-01 12:06:28 
E4_S126233 | estado: 0 | minutos: 136.5 

INICIANDO: E4_S877577 2026-11-01 14:22:56 
E4_S877577 | estado: 0 | minutos: 136.6 

INICIANDO: E4_S388403 2026-11-01 16:39:30 
E4_S388403 | estado: 0 | minutos: 138.9 

INICIANDO: E4_S308563 2026-11-01 18:58:25 
E4_S308563 | estado: 0 | minutos: 138.2 

INICIANDO: E4_S344785 2026-11-01 21:16:36 
E4_S344785 | estado: 0 | minutos: 133.1 

INICIANDO: E4_S618973 2026-11-01 23:29:42 
E4_S618973 | estado: 0 | minutos: 134.8 

INICIANDO: E4_S245039 2026-11-02 01:44:32 
E4_S245039 | estado: 0 | minutos: 130.7 

INICIANDO: E4_S552707 2026-11-02 03:55:13 
E4_S552707 | estado: 0 | minutos: 135.1 

FINALIZO


In [19]:
# ==========================================
# CONSOLIDAR RESULTADOS Y WILCOXON
# ==========================================

fs <- list.files(BASE, pattern = "^PARAM\\.yml$", recursive = TRUE, full.names = TRUE)

tb <- rbindlist(lapply(fs, function(f) {
  p <- read_yaml(f)
  data.table(
    id      = p$id_ejecucion,
    cfg     = p$configuracion,
    semilla = p$semilla_primigenia,
    gan     = p$resultado$ganancia_suavizada_max,
    envios  = p$resultado$envios
  )
}))

# tiempos: se toma el ultimo registro exitoso de cada corrida
if (file.exists(ARCHIVO_TIEMPOS)) {
  tt <- fread(ARCHIVO_TIEMPOS)[estado == 0, .(minutos = tail(minutos, 1)), by = id]
  tb <- merge(tb, tt, by = "id", all.x = TRUE)
} else tb[, minutos := NA_real_]

setorder(tb, cfg, semilla)
fwrite(tb, file.path(BASE, "consolidado_resultados.csv"))
print(tb)

print(tb[, .(n = .N, gan_media = mean(gan), gan_sd = sd(gan),
             min_medio = mean(minutos, na.rm = TRUE)), by = cfg])

# Wilcoxon pareado por semilla contra E0, con ajuste de Holm
w <- dcast(tb, semilla ~ cfg, value.var = "gan")
otras <- intersect(c("E1", "E2", "E3", "E4"), names(w))

if ("E0" %in% names(w) && length(otras) > 0) {
  n_pares <- sapply(otras, function(e) sum(complete.cases(w[, c("E0", e), with = FALSE])))
  pv <- sapply(otras, function(e) {
    ok <- complete.cases(w[, c("E0", e), with = FALSE])
    if (sum(ok) < 6) return(NA_real_)          # muy pocos pares para concluir
    wilcox.test(w[[e]][ok], w$E0[ok], paired = TRUE)$p.value
  })
  print(data.table(config = otras, n_pares = n_pares,
                   p_crudo = pv, p_holm = p.adjust(pv, method = "holm")))
}

Key: <id>
            id    cfg semilla     gan envios minutos
        <char> <char>   <num>   <num>  <int>   <num>
 1: E0_S126233     E0  126233 91.6975   1997    52.9
 2: E0_S216751     E0  216751 97.3225   2819    51.0
 3: E0_S245039     E0  245039 92.5975   1802    47.4
 4: E0_S308563     E0  308563 92.0875   1533    52.0
 5: E0_S344785     E0  344785 93.4450   1909    53.8
 6: E0_S388403     E0  388403 91.5800   2029    51.4
 7: E0_S552707     E0  552707 89.2250   2920    53.0
 8: E0_S618973     E0  618973 89.0875   1788    50.8
 9: E0_S770503     E0  770503 92.0450   1689    54.1
10: E0_S877577     E0  877577 92.9850   1861    51.2
11: E1_S126233     E1  126233 93.0550   1997    79.1
12: E1_S216751     E1  216751 99.1050   1994    79.7
13: E1_S245039     E1  245039 92.5300   1851    77.3
14: E1_S308563     E1  308563 96.3400   1769    77.8
15: E1_S344785     E1  344785 95.0675   2330    78.0
16: E1_S388403     E1  388403 96.0625   1770    82.3
17: E1_S552707     E1  552707 92.762